# Capitolo 8 — Autoencoder su MNIST: compressione, spazio latente, anomalie, VAE (§ 8.2–8.3, 8.5)
Circa 4 minuti in tutto su un portatile.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
fissa_seme(42)

from torchvision import datasets, transforms
from sklearn.metrics import roc_auc_score
tr = transforms.ToTensor()
mn_tr = datasets.MNIST("../data", train=True, download=True, transform=tr); mn_te = datasets.MNIST("../data", train=False, download=True, transform=tr)
fm_te = datasets.FashionMNIST("../data", train=False, download=True, transform=tr)
X_tr = mn_tr.data.float().div(255).view(-1, 784); X_te = mn_te.data.float().div(255).view(-1, 784); y_te = mn_te.targets.numpy(); X_fm = fm_te.data.float().div(255).view(-1, 784)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, d))
        self.dec = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    def forward(self, x): z = self.enc(x); return self.dec(z), z

def addestra_ae(modello, X, epoche=10, bs=128):
    opt = torch.optim.Adam(modello.parameters(), lr=1e-3); perdita_fn = nn.MSELoss()
    for epoca in range(epoche):
        modello.train(); perm = torch.randperm(len(X)); s = 0
        for i in range(0, len(X), bs):
            xb = X[perm[i:i + bs]]; opt.zero_grad(); perdita = perdita_fn(modello(xb)[0], xb); perdita.backward(); opt.step(); s += perdita.item() * len(xb)
        print(f"  epoca {epoca+1}: {s/len(X):.5f}")
    modello.eval(); return modello

def errore_ricostruzione(modello, X):
    modello.eval()
    with torch.no_grad(): return torch.cat([((modello(X[i:i+1000])[0] - X[i:i+1000]) ** 2).mean(1) for i in range(0, len(X), 1000)]).numpy()

fissa_seme(42); ae32 = addestra_ae(Autoencoder(32), X_tr)
fissa_seme(42); ae2 = addestra_ae(Autoencoder(2), X_tr)

## Confronto con la PCA

In [ ]:
mu = X_tr.mean(0); U, S, Vt = torch.linalg.svd(X_tr[:20000] - mu, full_matrices=False)
def pca(X, k): Z = (X - mu) @ Vt[:k].T; return Z @ Vt[:k] + mu, Z
for d, ae in ((32, ae32), (2, ae2)):
    print(f"{d:2d} dimensioni: autoencoder {errore_ricostruzione(ae, X_te).mean():.4f}   PCA {((pca(X_te, d)[0] - X_te) ** 2).mean():.4f}")
with torch.no_grad(): r32, r2 = ae32(X_te[:10])[0], ae2(X_te[:10])[0]
fig, ax = plt.subplots(5, 10, figsize=(10, 5))
for j in range(10):
    for r, (img, t) in enumerate(((X_te[j], "originale"), (pca(X_te[:10], 32)[0][j], "PCA 32"), (r32[j], "AE 32"), (pca(X_te[:10], 2)[0][j], "PCA 2"), (r2[j], "AE 2"))):
        ax[r, j].imshow(img.view(28, 28), cmap="gray", vmin=0, vmax=1); ax[r, j].axis("off")
        if j == 0: ax[r, j].set_title(t, loc="left", fontsize=8)
plt.show()

## Lo spazio latente a 2 dimensioni

In [ ]:
with torch.no_grad(): Z2 = ae2(X_te)[1].numpy()
plt.figure(figsize=(6, 5)); sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=y_te, cmap="tab10", s=3, alpha=0.6); plt.colorbar(sc, ticks=range(10)); plt.grid(alpha=0.3); plt.show()

## Anomalie: i capi di Fashion-MNIST visti dall'autoencoder delle cifre

In [ ]:
e_mn, e_fm = errore_ricostruzione(ae32, X_te), errore_ricostruzione(ae32, X_fm)
soglia = np.percentile(errore_ricostruzione(ae32, X_tr[:10000]), 99)
auc = roc_auc_score(np.r_[np.zeros(len(e_mn)), np.ones(len(e_fm))], np.r_[e_mn, e_fm])
print(f"errore medio cifre {e_mn.mean():.4f} | capi {e_fm.mean():.4f} | AUC {auc:.3f}")
print(f"soglia 99° percentile {soglia:.4f}: capi segnalati {(e_fm > soglia).mean():.1%}, cifre segnalate (falsi allarmi) {(e_mn > soglia).mean():.1%}")
plt.hist(e_mn, bins=80, range=(0, 0.08), alpha=0.6, density=True, label="cifre"); plt.hist(e_fm, bins=80, range=(0, 0.08), alpha=0.6, density=True, label="capi"); plt.axvline(soglia, ls="--", color="k"); plt.legend(); plt.xlabel("errore di ricostruzione"); plt.show()

## Il VAE, in breve

In [ ]:
class VAE(nn.Module):
    def __init__(self, d=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU()); self.mu = nn.Linear(64, d); self.logvar = nn.Linear(64, d)
        self.dec = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    def forward(self, x):
        h = self.enc(x); mu, lv = self.mu(h), self.logvar(h); z = mu + torch.randn_like(mu) * torch.exp(0.5 * lv); return self.dec(z), mu, lv
fissa_seme(42); vae = VAE(); opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
for epoca in range(10):
    vae.train(); perm = torch.randperm(len(X_tr))
    for i in range(0, len(X_tr), 128):
        xb = X_tr[perm[i:i + 128]]; opt.zero_grad(); r, mu_, lv_ = vae(xb)
        ricostruzione = nn.functional.binary_cross_entropy(r, xb, reduction="sum") / len(xb); kl = -0.5 * (1 + lv_ - mu_ ** 2 - lv_.exp()).sum(1).mean()
        (ricostruzione + kl).backward(); opt.step()
    print(f"  epoca {epoca+1}: ricostruzione {ricostruzione.item():.1f}  KL {kl.item():.1f}")
vae.eval(); fissa_seme(0)
with torch.no_grad(): campioni = vae.dec(torch.randn(40, 16)).view(4, 10, 28, 28)
plt.figure(figsize=(10, 4)); plt.imshow(np.vstack([np.hstack([campioni[i, j] for j in range(10)]) for i in range(4)]), cmap="gray"); plt.axis("off"); plt.title("cifre che non esistono: z ~ N(0, I) decodificato"); plt.show()